Define static or baseline covariates, get a wide-format file for the cohort including important covariates. 

# Setup

In [ ]:
library(tidyverse)
library(bigrquery)
library(ggplot2)
library(data.table)

In [ ]:
source("functions.R")

In [ ]:
tic("Time to run script")

# Demographics 

In [ ]:
demog_sql <- paste("
    SELECT
        person.BIRTH_DATETIME as date_of_birth,
        person.person_id,
        p_race_concept.concept_name as race,
        p_gender_concept.concept_name as gender,
        p_ethnicity_concept.concept_name as ethnicity,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            on person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_gender_concept 
            on person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            on person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            on person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
demog_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  #strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_22125675",
  "demog.csv")
message(str_glue('The data will be written to {demog_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), demog_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  demog_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_22125675_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(gender = col_character(), race = col_character(), ethnicity = col_character(), sex_at_birth = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
demog_df <- read_bq_export_from_workspace_bucket(demog_path)

dim(demog_df)

In [ ]:
demog_df %>% count(race, sort=T)
demog_df %>% count(ethnicity, sort=T)
demog_df %>% count(sex_at_birth, sort=T)
demog_df %>% count(gender, sort=T)

In [ ]:
makeNA <- function(x) {
        madeNA <- x
        madeNA[x %in% c("PMI: Prefer Not To Answer", "PMI: Skip", #"What Race Ethnicity: Race Ethnicity None Of These",
                            "No matching concept", "Not male, not female, prefer not to answer, or skipped", 
                            "Not man only, not woman only, prefer not to answer, or skipped", #"Another single population",
                            "I prefer not to answer", "None Indicated", "Dont Know", "Skip")] <- NA
    return(madeNA)
}

demog_clean <- demog_df %>%
    ungroup() %>%
    mutate(date_of_birth = as.Date(date_of_birth),
          race = makeNA(race),
          ethnicity = makeNA(ethnicity),
          sex_at_birth = makeNA(sex_at_birth),
          gender = makeNA(gender)
          ) %>%
    select(person_id, date_of_birth, everything()) %>%
    arrange(person_id) %>%
    mutate(ethnicity = gsub("What Race Ethnicity: Race Ethnicity ", "", ethnicity), 
          gender = gsub("Gender Identity: ", "", gender), 
          sex_at_birth = gsub("Sex At Birth: Sex At Birth ", "", sex_at_birth))

In [ ]:
demog_clean %>% count(race, sort=T)
demog_clean %>% count(ethnicity, sort=T)
demog_clean %>% count(sex_at_birth, sort=T)
demog_clean %>% count(gender, sort=T)

In [ ]:
suppressWarnings(rm(demog_df, demog_path, demog_sql))
gc()

# Survey data

## Read in survey data

In [ ]:
library(tidyverse)
library(bigrquery)

survey_sql <- paste("
    SELECT
        answer.question,
        answer.answer,
        answer.survey_datetime,
        answer.person_id,
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (
                SELECT
                    DISTINCT concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (
                        select
                            cast(cr.id as string) as id 
                        FROM
                            `cb_criteria` cr 
                        WHERE
                            concept_id IN (
                                1586134,1585855,1585710
                            ) 
                            AND domain_id = 'SURVEY'
                    ) a 
                        ON (
                            c.path like CONCAT('%',
                        a.id,
                        '.%')) 
                    WHERE
                        domain_id = 'SURVEY' 
                        AND type = 'PPI' 
                        AND subtype = 'QUESTION'
                    )
            )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  #strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "survey_22125675",
  "survey*.csv")
message(str_glue('The data will be written to {survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_22125675_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
survey_df <- read_bq_export_from_workspace_bucket(survey_path)

dim(survey_df)
length(unique(survey_df$person_id))

Choose which survey questions we are interested in. 

In [ ]:
unique(survey_df$question)

In [ ]:
questions_keep <- c('Organ Transplant: Organ Transplant Description', 
                           'Income: Annual Income', 
                           'Marital Status: Current Marital Status', 
                           'Insurance: Health Insurance', 
                           'Health Insurance: Health Insurance Type', 
                           'Health Insurance: Insurance Type Update',
                    'Active Duty: Active Duty Serve Status',
                           'Smoking: 100 Cigs Lifetime',
                           'Attempt Quit Smoking: Completely Quit Age',
                           'Smoking: Average Daily Cigarette Number', 
                           'Smoking: Current Daily Cigarette Number',
                           'Smoking: Smoke Frequency', 
                           'Smoking: Number Of Years', 
                           'Smoking: Daily Smoke Starting Age', 
                           'Education Level: Highest Grade', 
                           'Alcohol: Alcohol Participant',
                           'Alcohol: Drink Frequency Past Year', 
                           'Alcohol: Average Daily Drink Count', 
                           'Alcohol: 6 or More Drinks Occurrence', 
                           'Employment: Employment Status',  
                           'Overall Health: General Health', 
                           'Overall Health: General Quality', 
                           'Overall Health: General Physical Health', 
                           'Overall Health: Average Pain 7 Days')
length(questions_keep)

survey_clean <- survey_df %>%
    filter(question %in% questions_keep) %>%
    mutate(question = gsub("^.*\\: | ", "", question)) %>%
    mutate(across(answer, makeNA)) %>%
    mutate(answer = gsub("^.*\\: ", "", answer)) %>%
    mutate(across(answer, makeNA))

length(unique(survey_clean$question))

In [ ]:
survey_clean %>% filter(question == "ActiveDutyServeStatus") %>% count(answer, sort=T)

## Process kidney transpant survey info

In [ ]:
survey_clean %>% filter(question == 'OrganTransplantDescription') %>%
    count(answer, sort=T) 

In [ ]:
survey_clean <- survey_clean %>%
    filter(!(question == 'OrganTransplantDescription' & answer != "Kidney")) %>%
    mutate(question = ifelse(question == 'OrganTransplantDescription', "KidneyTransplant_survey", question)) %>%
    mutate(answer = ifelse(question == "KidneyTransplant_survey" & answer == "Kidney", "1", answer))

## Process Health Insurance information, create flag for possible VA patients

Note that question about active duty status is available in the controlled tier but suppressed in the registered tier. 

In [ ]:
insurance_types <- survey_clean %>%
    filter(question %in% c('HealthInsuranceType', 'InsuranceTypeUpdate'))

survey_clean <- survey_clean %>%
    anti_join(insurance_types)

In [ ]:
insurance_types %>% count(answer, sort=T)

In [ ]:
# See all the combinations of insurance types: each individual can select multiple
all_insurance_types <- insurance_types %>%
    filter(!is.na(answer)) %>%
    mutate(answer = ifelse(answer == "None", "No Coverage", answer)) %>%
    group_by(person_id) %>%
    summarize(insurance_types = paste0(sort(unique(answer)), collapse = ", ")) %>%
    ungroup()

In [ ]:
# This is *almost* consistent
all_insurance_types %>% count(insurance_types, sort=T) %>% filter(grepl("No Coverage", insurance_types))

In [ ]:
# Define a few flags that may be useful
# These flags are separate because more than 1 can be true

Medicare <- insurance_types %>% filter(answer == "Medicare") %>% 
    distinct(person_id) %>% mutate(Medicare = 1)
Medicaid <- insurance_types %>% filter(answer == "Medicaid") %>% 
    distinct(person_id) %>% mutate(Medicaid = 1)
MilitaryHealthInsurance <- insurance_types %>% filter(answer %in% c("VA", "Military")) %>% 
    distinct(person_id) %>% mutate(MilitaryHealthInsurance = 1)
Uninsured <- insurance_types %>% filter(answer %in% c("None", "No Coverage")) %>% 
    distinct(person_id) %>% mutate(Uninsured = 1)
PrivateHealthInsurance <- insurance_types %>% filter(answer %in% c("Employer Or Union", "Private")) %>% 
    distinct(person_id) %>% mutate(PrivateHealthInsurance = 1)
OtherHealthInsurance <- insurance_types %>% filter(answer %in% c("Purchased", "Other Health Plan", "State Sponsored", 
                            "Single Service", "Medi GAP", "Other Government", "Indian", "Schip")) %>% 
    distinct(person_id) %>% mutate(OtherHealthInsurance = 1)

insurance_flags <- full_join(Medicare, Medicaid) %>%
    full_join(MilitaryHealthInsurance) %>%
    full_join(Uninsured) %>%
    full_join(PrivateHealthInsurance) %>%
    full_join(OtherHealthInsurance)

# fill in NAs with 0
insurance_flags[is.na(insurance_flags)] <- 0

In [ ]:
nrow(insurance_flags)
apply(insurance_flags, 2, mean) %>% signif(2)

## Process Employment field
Participants were allowed to check multiple boxes on the same survey. 

In [ ]:
employment_df <- survey_clean %>% filter(question == "EmploymentStatus")

In [ ]:
employment_df %>% count(answer, sort=T)

In [ ]:
#patients don't have more than one survey date
employment_df %>% group_by(person_id) %>% 
    filter(length(unique(survey_datetime)) > 1) %>% 
    arrange(person_id, survey_datetime) 

In [ ]:
employment <- employment_df %>%
    filter(!is.na(answer)) %>%
    mutate(answer = 
           case_when(answer %in% c("Employed For Wages", "Self Employed") ~ "Employed",
                     answer %in% c("Unable To Work", "Out Of Work One Or More", 
                                   "Out Of Work Less Than One", "Homemaker") ~ "Unemployed",
                     TRUE ~ answer
                             )) %>% distinct()

employment %>% count(answer, sort=T)

In [ ]:
# Some gave multiple answers
employed_multi_summ <- employment %>%
    group_by(person_id, question, survey_datetime) %>%
    arrange(person_id, survey_datetime, answer) %>%
    summarise(answer = paste0(answer, collapse = ", ")) %>%
    ungroup() 

employed_multi_summ %>% count(answer, sort=T)

In [ ]:
# Prioritize answers so that we can collapse to 1 variable
employment_multi_fix <- employed_multi_summ %>%
    mutate(answer = case_when(answer == "Employed, Student" ~ "Student",
                              answer == "Employed, Retired" ~ "Retired",
                              answer == "Retired, Unemployed" ~ "Retired",
                              answer == "Employed, Unemployed" ~ "Employed",
                              answer == "Student, Unemployed" ~ "Student",
                              answer == "Employed, Student, Unemployed" ~ "Student",
                              answer == "Employed, Retired, Unemployed" ~ "Retired",
                              answer == "Retired, Student" ~ "Retired",
                              answer == "Retired, Student, Unemployed" ~ "Retired",
                              answer == "Employed, Retired, Student" ~ "Retired",
                              answer == "Employed, Retired, Student, Unemployed" ~ "Retired",
                              TRUE ~ answer
                             ))

employment_multi_fix %>% count(answer, sort=T)

In [ ]:
# replace with the cleaned version
survey_clean <- survey_clean %>%
    anti_join(employment_df) %>% 
    full_join(employment_multi_fix) 

In [ ]:
# Make sure there are no more duplicates left.  Want only 1 answer per question per person per survey date. 
survey_clean %>% group_by(person_id, survey_datetime, question) %>% filter(n() > 1) 

Now that there are no fields with more than one answer, we can convert the survey data to wide format. 

In [ ]:
survey_first_datetime <- survey_clean %>%
    mutate(survey_datetime = as.Date(survey_datetime)) %>%
    group_by(person_id) %>%
    summarize(survey_first_datetime = min(survey_datetime)) %>%
    ungroup()

survey_wide <- survey_clean %>%
    select(-survey_datetime) %>%
    distinct() %>%
    full_join(survey_first_datetime) %>%
    pivot_wider(names_from = "question", values_from = "answer") %>%
    select(person_id, survey_first_datetime, everything())

In [ ]:
#Fix some data types and insurance flags
survey_wide <- survey_wide %>%
    mutate(across(c(AverageDailyCigaretteNumber, CurrentDailyCigaretteNumber, CompletelyQuitAge,
                  DailySmokeStartingAge, AveragePain7Days, NumberOfYears, KidneyTransplant_survey),
                  as.numeric)) %>%
    full_join(insurance_flags) %>%
    mutate(KidneyTransplant_survey = coalesce(KidneyTransplant_survey, 0)) %>% 
    mutate(Uninsured = ifelse(!is.na(HealthInsurance) & HealthInsurance == "No", 1, Uninsured)) %>%
    mutate(PrivateHealthInsurance = ifelse(!is.na(Uninsured) & is.na(PrivateHealthInsurance), 0, PrivateHealthInsurance), 
           OtherHealthInsurance = ifelse(!is.na(Uninsured) & is.na(OtherHealthInsurance), 0, OtherHealthInsurance),
           Medicare = ifelse(!is.na(Uninsured) & is.na(Medicare), 0, Medicare),
           Medicaid = ifelse(!is.na(Uninsured) & is.na(Medicaid), 0, Medicaid),
          ) %>%
    mutate(Uninsured = ifelse(!is.na(Uninsured) & Uninsured == 1 &  
                              ( (!is.na(Medicare) & Medicare == 1) |
                               (!is.na(Medicaid) & Medicaid == 1) |
                               (!is.na(PrivateHealthInsurance) & PrivateHealthInsurance == 1) |
                               (!is.na(OtherHealthInsurance) & OtherHealthInsurance == 1) 
                              ),
                             0, Uninsured))

## Process Smoking-related fields

In [ ]:
# Take all the smoking survey answers and create 2 variables.  A 2-level factor (smoker nonsmoker), and a 3-level one
# (smoker, former smoker, current smoker)

survey_wide <- survey_wide %>%
    mutate(smoke_ever = 
           ifelse( `100CigsLifetime` == "No" & 
                  is.na(SmokeFrequency) & 
                  (is.na(AverageDailyCigaretteNumber) | AverageDailyCigaretteNumber == 0) & 
                  (is.na(CurrentDailyCigaretteNumber) | CurrentDailyCigaretteNumber == 0) & 
                  (is.na(CompletelyQuitAge)) & 
                  (is.na(DailySmokeStartingAge)) & 
                  (is.na(NumberOfYears) | NumberOfYears == 0), 
                  0, #Non-smoker
              ifelse(`100CigsLifetime` == "Yes" | 
                     (!is.na(SmokeFrequency) & SmokeFrequency %in% c("Every Day", "Some Days")) |
                     (!is.na(AverageDailyCigaretteNumber) & AverageDailyCigaretteNumber > 0) | 
                     (!is.na(CurrentDailyCigaretteNumber) & CurrentDailyCigaretteNumber > 0) |
                     !is.na(CompletelyQuitAge) | 
                     !is.na(DailySmokeStartingAge) |
                     (!is.na(NumberOfYears) & NumberOfYears > 0),
                     1, #ever smokers
                     NA))) %>% #can't be categorized
    mutate(smoke_category = ifelse(!smoke_ever, "Never Smoker", 
                              ifelse(SmokeFrequency == "Not At All", "Former Smoker", 
                                ifelse(SmokeFrequency %in% c("Every Day", "Some Days"), "Current Smoker", NA))))

survey_wide %>% 
    select(`100CigsLifetime`, SmokeFrequency, smoke_ever, smoke_category) %>%
    group_by_all() %>%
    count() %>%
    arrange(desc(n)) 

In [ ]:
# pack-years smoking
survey_wide <- survey_wide %>% 
    mutate(calc.yearssmoke = CompletelyQuitAge - DailySmokeStartingAge) %>%
    mutate(calc.yearssmoke = ifelse(calc.yearssmoke > 0, calc.yearssmoke, NA)) %>%
    mutate(diff = calc.yearssmoke - NumberOfYears,
          years_for_packyears = coalesce(NumberOfYears, calc.yearssmoke),
          cigs_for_packyears = coalesce(AverageDailyCigaretteNumber, CurrentDailyCigaretteNumber),
           packyears_smoke = years_for_packyears * (cigs_for_packyears / 20) ) %>% #20 cigs per pack
    mutate(packyears_smoke = ifelse(!is.na(packyears_smoke), packyears_smoke,
                                   ifelse(!smoke_ever, 0, NA)))

In [ ]:
survey_wide <- survey_wide %>% 
    select(-c(calc.yearssmoke, diff, years_for_packyears, cigs_for_packyears)) %>%
    dplyr::rename(
        Cigs100Lifetime = `100CigsLifetime`,
        Occurrence6orMoreDrinks = `6orMoreDrinksOccurrence`,
        NumberOfYearsSmoked = NumberOfYears
    ) 

In [ ]:
survey_wide <- survey_wide %>% 
    arrange(person_id) %>% ungroup() %>%
    select(person_id, survey_first_datetime, 
           contains(c("smoke", "quit", "cig", "alcohol", "drink", "insur", "medi")),
           everything()) # reorder variables

In [ ]:
suppressWarnings(rm(survey_df, survey_clean, survey_first_datetime, survey_path, survey_sql, employment, 
                    employment_bad, employment_fixed, all_insurance_types, chk, employed_multi_summ, employment_df,
                   employment_multi_fix, insurance_flags, insurance_types, Medicaid, Medicare, MilitaryHealthInsurance, 
                   OtherHealthInsurance, PrivateHealthInsurance, Uninsured))
gc()

# Define visit types and extract outpatient visit dates

These may be used for defining censoring, filtering biomarkers, defining newly diagnosed diabetes, measures of healthcare utilization, etc. 

## Define visit types 

In [ ]:
# Get the distinct visit concept IDs and what they mean
visit_types_sql <- paste0("
    SELECT DISTINCT      
        v.visit_concept_id,
        concept.concept_name as visit_concept_name,
    FROM `visit_occurrence` v
    LEFT JOIN `concept` concept 
        on v.VISIT_CONCEPT_ID = concept.CONCEPT_ID
    ")

visit_types_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  #strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "handwritten",
  "visit_types.csv")

bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), visit_types_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  visit_types_path,
  destination_format = "CSV")

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
visit_types_df <- read_bq_export_from_workspace_bucket(visit_types_path)

dim(visit_types_df)

visit_types_df

In [ ]:
outpat_visit_types <- visit_types_df %>%
    filter(grepl("outpat|ambulatory clinic|office visit|Health examination|comprehensive preventive medicine|Primary Care Clinic|Federally Qualified Health Center", 
                 visit_concept_name, ignore.case = TRUE)) %>%
    filter(!grepl("Rehab", visit_concept_name))

home_visit_types <- visit_types_df %>%
    anti_join(outpat_visit_types) %>%
    filter(grepl("home", visit_concept_name, ignore.case = TRUE)) 

ER_urgentcare_visit_types <- visit_types_df %>%
    anti_join(outpat_visit_types) %>%
    anti_join(home_visit_types) %>%
    filter(grepl("emergency|urgent|ambulance", visit_concept_name, ignore.case = TRUE)) %>%
    filter(!grepl("non-emergen", visit_concept_name, ignore.case = TRUE))

inpatient_visit_types <- visit_types_df %>%
    anti_join(outpat_visit_types) %>%
    anti_join(home_visit_types) %>%
    anti_join(ER_urgentcare_visit_types) %>%
    filter(grepl("inpatient|intensive|hospital", visit_concept_name, ignore.case = TRUE)) %>%
    filter(!grepl("non-hospital", visit_concept_name, ignore.case = TRUE))

telehealth_visit_types <- visit_types_df %>%
    anti_join(outpat_visit_types) %>%
    anti_join(home_visit_types) %>%
    anti_join(ER_urgentcare_visit_types) %>%
    anti_join(inpatient_visit_types) %>%
    filter(grepl("telehealth", visit_concept_name, ignore.case = TRUE))

observation_room_visit_types <- visit_types_df %>%
    anti_join(outpat_visit_types) %>%
    anti_join(home_visit_types) %>%
    anti_join(ER_urgentcare_visit_types) %>%
    anti_join(inpatient_visit_types) %>%
    anti_join(telehealth_visit_types) %>%
    filter(grepl("observation", visit_concept_name, ignore.case = TRUE)) %>%
    filter(!grepl("Drug test", visit_concept_name, ignore.case = TRUE)) 

specialty_visit_types <- visit_types_df %>%
    anti_join(outpat_visit_types) %>%
    anti_join(home_visit_types) %>%
    anti_join(ER_urgentcare_visit_types) %>%
    anti_join(inpatient_visit_types) %>%
    anti_join(telehealth_visit_types) %>%
    anti_join(observation_room_visit_types)

In [ ]:
outpat_visit_types

In [ ]:
home_visit_types

In [ ]:
ER_urgentcare_visit_types

In [ ]:
inpatient_visit_types

In [ ]:
telehealth_visit_types

In [ ]:
observation_room_visit_types

In [ ]:
specialty_visit_types_lab <- specialty_visit_types %>%
    filter(grepl("Lab|metabolic panel", visit_concept_name, ignore.case=T))
specialty_visit_types_lab

In [ ]:
specialty_visit_types_pharmacy <- specialty_visit_types %>%
    filter(grepl("Pharmacy", visit_concept_name, ignore.case=T))
specialty_visit_types_pharmacy

In [ ]:
specialty_visit_types_ambulatory  <- specialty_visit_types %>%
    filter(grepl("ambulatory|outpatient|mammography|Immunization Center", visit_concept_name, ignore.case=T))
specialty_visit_types_ambulatory

In [ ]:
specialty_visit_types_other  <- specialty_visit_types %>%
    anti_join(specialty_visit_types_lab) %>%
    anti_join(specialty_visit_types_pharmacy) %>%
    anti_join(specialty_visit_types_ambulatory)

specialty_visit_types_other

In [ ]:
visit_types_key <- rbind(outpat_visit_types %>% mutate(visit_type = "outpatient_visit"),
                         home_visit_types %>% mutate(visit_type = "home_visit"),
                         ER_urgentcare_visit_types %>% mutate(visit_type = "ER_or_urgent_care_visit"),
                         inpatient_visit_types  %>% mutate(visit_type = "inpatient_visit"),
                         telehealth_visit_types %>% mutate(visit_type = "telehealth_visit"),
                         observation_room_visit_types  %>% mutate(visit_type = "observation_room_visit"),
                         specialty_visit_types_lab  %>% mutate(visit_type = "lab_visit"),
                         specialty_visit_types_pharmacy  %>% mutate(visit_type = "pharmacy_visit"),
                         specialty_visit_types_ambulatory  %>% mutate(visit_type = "specialty_ambulatory_visit"),
                         specialty_visit_types_other  %>% mutate(visit_type = "other_unknown_visit")
                        )
visit_types_key

In [ ]:
write_to_bucket(visit_types_key, "visit_types_key.csv") 

In [ ]:
#Use these for the next query
visit_types_key %>% filter(visit_type %in% c("outpatient_visit", "home_visit")) %>%
    distinct(visit_concept_id) %>% .$visit_concept_id %>% cat()

In [ ]:
suppressWarnings(rm(outpat_visit_types, home_visit_types, ER_urgentcare_visit_types, inpatient_visit_types, 
                    telehealth_visit_types, observation_room_visit_types, specialty_visit_types_lab, 
                    specialty_visit_types_pharmacy, specialty_visit_types_ambulatory, specialty_visit_types_other,  
                    specialty_visit_types, visit_types_df, visit_types_path, visit_types_sql))

## Extract all outpatient and home visit dates. 

In [ ]:
# This query was handwritten to get newly DmDx status
all_outpat_home_visits_sql <- paste0("
    SELECT DISTINCT      
        cb_search_all_events.person_id,
        cb_search_all_events.entry_date as date_outpat_or_home_visit
    FROM
        cb_search_all_events
    WHERE CONCEPT_ID IN (8966, 32261, 8756, 32693, 38004247, 2514527, 2514520, 2514528, 38004207, 
                        581477, 9202, 2514521, 581476, 38004519)                          
                                AND is_standard = 1 
    ")

all_outpat_home_visits_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  #strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "handwritten",
  "all_outpat_home_visits*.csv")
message(str_glue('The data will be written to {all_outpat_home_visits_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), all_outpat_home_visits_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  all_outpat_home_visits_path,
  destination_format = "CSV")

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
all_outpat_home_visits_df <- read_bq_export_from_workspace_bucket(all_outpat_home_visits_path)
dim(all_outpat_home_visits_df)

In [ ]:
write_to_bucket(all_outpat_home_visits_df, "all_outpatient_and_home_visit_dates.csv")

# Extract last visit, death information

These may be used for defining censoring and/or competing risk outcomes. 

## Last outpatient or home visit 

In [ ]:
last_outpx <- all_outpat_home_visits_df %>%
    group_by(person_id) %>%
    summarize(last_outpat_or_home_visit = max(date_outpat_or_home_visit))

In [ ]:
suppressWarnings(rm(all_outpat_home_visits_df, all_outpat_home_visits_path, all_outpat_home_visits_sql))
gc()

## Last visit of any type

In [ ]:
last_visit_any_sql <- paste0("
    SELECT DISTINCT      
        cb_search_all_events.person_id,
         max(cb_search_all_events.entry_date) as last_visit_any
    FROM
        cb_search_all_events
    WHERE is_standard = 1 
    GROUP BY person_id
    ")

last_visit_any_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "data/Feb2025",
  "last_visit_any.csv")
last_visit_any_path

bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), last_visit_any_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  last_visit_any_path,
  destination_format = "CSV")

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
last_visit_any_df <- read_bq_export_from_workspace_bucket(last_visit_any_path)

In [ ]:
rm(last_visit_any_path, last_visit_any_sql)

## Death

In [ ]:
death_sql <- "SELECT DISTINCT 
    person_id, 
    death_date,
    death_type.concept_name as death_type_concept_name,
    cause_concept_id,
    cause_source_value,
    cause_source_concept_id,
    cause.concept_name as cause_concept_name,
FROM `death` d
LEFT JOIN `concept` death_type
    on d.death_type_concept_id = death_type.concept_id
LEFT JOIN `concept` cause
    on d.cause_concept_id = cause.concept_id"
death_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
    "death_*.csv")
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), death_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  death_path,
  destination_format = "CSV")

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(death_date = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
death_df <- read_bq_export_from_workspace_bucket(death_path) 

In [ ]:
death <- death_df %>% distinct(person_id, death_date, death_type_concept_name) %>%
    mutate(death_date = as.Date(death_date)) %>%
    group_by(person_id) %>% 
    slice_min(death_date) %>%
    distinct() %>%
    ungroup()

In [ ]:
death %>% count(death_type_concept_name, sort=T)

In [ ]:
suppressWarnings(rm(death_sql, death_path))

Note: there is also the aou_death table.  It has the primary_death_record flag.  If TRUE, the record appears in the main DEATH table.  

# Baseline Physical examination data

Some pariticipants elected to participate. 

In [ ]:
library(tidyverse)
library(bigrquery)

physical_sql <- paste("
    SELECT
        measurement.measurement_datetime,
        measurement.person_id,
        measurement.value_as_number,
        m_unit.concept_name as unit_concept_name,
        m_standard_concept.concept_name as standard_concept_name 
    FROM
        ( SELECT
            * 
        from
            `measurement` measurement 
        WHERE
            (
                measurement_source_concept_id IN  (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            select
                                cast(cr.id as string) as id 
                            FROM
                                `cb_criteria` cr 
                            WHERE
                                concept_id IN (
                                    1586218, 903107, 903109, 903110, 903111, 903112, 903115, 903117, 903118, 903121, 
                                    903124, 903126, 903127, 903132, 903133, 903135, 903136
                                ) 
                                AND full_text LIKE '%_rank1]%'
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1
                        )
                )
            ) measurement 
        left join
            `concept` m_unit 
                on measurement.unit_concept_id = m_unit.concept_id 
        left join
            `concept` m_standard_concept 
                on measurement.measurement_concept_id = m_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
physical_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  #strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "measurement_94118835",
  "physical*.csv")

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), physical_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  physical_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurement_94118835_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), unit_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
physical_df <- read_bq_export_from_workspace_bucket(physical_path)

In [ ]:
physical_df <- physical_df %>% 
    filter(!(grepl("mean|panel", standard_concept_name))) %>%
    mutate(value_as_number = ifelse(standard_concept_name == "Participant is a wheelchair user", 1, value_as_number)) %>%
    filter(!is.na(value_as_number)) %>%
    mutate(measurement_date = as.Date(measurement_datetime)) %>%
    distinct() %>%
    group_by(person_id, standard_concept_name) %>%
    slice_min(measurement_date) %>% #get the first non-missing value of each measure
    ungroup() %>%
    select(-measurement_datetime)

In [ ]:
physical_first <- physical_df %>%
    group_by(person_id) %>%
    summarize(physical_exam_date = min(measurement_date))

In [ ]:
physical_wide <- physical_df %>%
    select(-measurement_date, -unit_concept_name) %>%
    distinct() %>%
    pivot_wider(names_from = "standard_concept_name", values_from = "value_as_number",
               values_fn = function(x) median(x, na.rm=T)) %>%
    left_join(physical_first) %>%
    select(person_id, physical_exam_date, everything())

In [ ]:
names(physical_wide) <- gsub(" |-", "_", names(physical_wide))
names(physical_wide) <- gsub("___", "_", names(physical_wide))
names(physical_wide)[5] <- "BMI"

In [ ]:
physical_wide <- physical_wide %>%
    mutate(Participant_is_a_wheelchair_user = coalesce(Participant_is_a_wheelchair_user, 0)) %>%
    dplyr::rename(waist_circumference = Adult_Waist_Circumference_Protocol, 
                  hip_circumference = PhenX_hip_circumference_protocol_020801)           

In [ ]:
suppressWarnings(rm(physical_df, physical_first, physical_path, physical_sql))
gc()

# Combine data and write out dataset

In [ ]:
suppressMessages(
covar_dat <- left_join(demog_clean, survey_wide) %>%
    left_join(last_visit_any_df) %>%
    left_join(last_outpx) %>%
    left_join(death) %>%
    left_join(physical_wide) %>%
    ungroup() %>%
    mutate(death = as.numeric(!is.na(death_date))) %>%
    select(person_id, date_of_birth, death, death_date, everything())
)

In [ ]:
# Get percentage non-NA
apply(covar_dat, 2, function(x) signif(mean(!is.na(x))*100, 3))

In [ ]:
write_to_bucket(covar_dat, "covariates_wide_all_participants.csv")

# Clean up

In [ ]:
toc()

In [ ]:
rm(list = ls())
gc()